In [0]:
df = spark.read.csv("/Volumes/lending-club-loan/default/data-lending-club-loan")
display(df)

### Reading a CSV file from volumes
Adding options:
1. inferSchema to true which automatically helps to update the data types.
2. The first row is taken as the headers.
3. multiLine and escape so quoted free-text fields (commas, line breaks, doubled quotes) stay in one row.

In [0]:
df = spark.read\
.option("inferSchema", "true")\
.option("header", "true")\
.option("multiLine", "true")\
.option("escape", "\"")\
.csv("/Volumes/lending-club-loan/default/data-lending-club-loan")
display(df)

### Atlering the Data Types of the columns
1. We are first capturing the schema from reading the CSV file.

In [0]:
from pyspark.sql import functions as F, types as T

temp_schema = spark.read\
  .option("header", "true")\
  .option("multiLine", "true")\
  .option("escape", "\"")\
  .option("inferSchema", "true")\
  .csv("/Volumes/lending-club-loan/default/data-lending-club-loan").schema

1. Now on the keeping the whole schema as it is and just changing the datatype.

2. Then, send the schema into the read.csv() option.

In [0]:
# Start from the full inferred schema and replace only the columns we want to alter
full_schema = T.StructType([
    T.StructField("id", T.IntegerType(), True) if f.name == "id"
    else T.StructField("annual_inc", T.DoubleType(), True) if f.name == "annual_inc"
    else f
    for f in temp_schema
])

altered_df = spark.read\
.option("header", "true")\
.option("multiLine", "true")\
.option("escape", "\"")\
.schema(full_schema)\
.csv("/Volumes/lending-club-loan/default/data-lending-club-loan")
display(altered_df)

In [0]:
display(altered_df.printSchema())

1. Renaming the terms column with term_in_months
2. We removed the months text from the column because numbers are more like to be useful.
3. Changed the date format for cloumns "issue_d", "earliest_cr_line", and "last_credit_pull_d".

In [0]:
from pyspark.sql import functions as F

altered_df = altered_df.withColumnRenamed("term", "term_in_months")

altered_df = altered_df.withColumn("term_in_months", F.regexp_replace("term_in_months", " months", "").cast("int"))

altered_df = altered_df.withColumn(
    "issue_d", 
    F.date_format(F.try_to_date(F.col("issue_d"), "MMM-yyyy"), "MM-yyyy")
)
altered_df = altered_df.withColumn(
    "earliest_cr_line", 
    F.date_format(F.try_to_date(F.col("earliest_cr_line"), "MMM-yyyy"), "MM-yyyy")
)
altered_df = altered_df.withColumn(
    "last_credit_pull_d", 
    F.date_format(F.try_to_date(F.col("last_credit_pull_d"), "MMM-yyyy"), "MM-yyyy")
)
display(altered_df)

In [0]:
from pyspark.sql.functions import col

annual_income_btw_5000_10000 = altered_df.filter((col("annual_inc") > 5000) & (col("annual_inc") < 10000))
display(annual_income_btw_5000_10000)

Basic summary, number of columns in the whole dataset

In [0]:
altered_df.count()

In [0]:
display(altered_df.summary())

Checking the number of null values according to each column.

In [0]:
from pyspark.sql import functions as F, types as T

# Counts both nulls and NaN values across the entire DataFrame
display(altered_df.select([F.count(F.when(F.col(c).isNull() | (F.isnan(c) if isinstance(altered_df.schema[c].dataType, T.NumericType) else F.lit(False)), c)).alias(c) for c in altered_df.columns]).show())

### Based on the busniess problem we are trying to solve we will only select the required columns.

In [0]:
keep_cols = [
    "id", "issue_d", "addr_state",
    "loan_status",
    "loan_amnt", "term_in_months", "int_rate", "installment",
    "grade", "sub_grade", "purpose", "application_type",
    "annual_inc", "emp_title", "emp_length", "home_ownership", "verification_status", "dti",
    "fico_range_low", "fico_range_high", "earliest_cr_line", "delinq_2yrs", "inq_last_6mths",
    "open_acc", "total_acc", "pub_rec", "pub_rec_bankruptcies", "revol_bal", "revol_util",
    "acc_now_delinq", "collections_12_mths_ex_med", "tax_liens",
    "tot_cur_bal", "tot_hi_cred_lim", "total_rev_hi_lim", "avg_cur_bal",
    "bc_open_to_buy", "bc_util", "mort_acc", "acc_open_past_24mths",
    "num_sats", "num_actv_rev_tl", "pct_tl_nvr_dlq", "percent_bc_gt_75",
    "mo_sin_old_rev_tl_op", "total_bal_ex_mort", "total_bc_limit",
    "mths_since_last_delinq", "mths_since_last_record", "mths_since_last_major_derog",
    "mths_since_recent_revol_delinq", "mths_since_recent_bc_dlq", "mths_since_recent_inq",
]

selected_df = altered_df.select(*keep_cols)
display(selected_df)


Checking distinct employee length

In [0]:
# Run this to see all unique values in the column
display(selected_df.select("emp_length").distinct())

Checking the count for the " reactors" " in the whole dataset.

In [0]:
from pyspark.sql.functions import col
emp_length_filter = selected_df.filter((col("emp_length").contains("reactors")))
display(emp_length_filter.select("emp_length").count())

emp_length_nulls = selected_df.filter(col("emp_length").isNull())
print("\nNull values: ", emp_length_nulls.count())

Replacing or transforming all the data with numeric such 10+ years to 10, 4 years to 4 and reactors" to 0.

In [0]:
selected_df = selected_df.withColumn(
    "emp_length",
    F.when(F.col("emp_length").contains("reactors"), "0")
    .when(F.col("emp_length").contains("< 1"), "0")
    .otherwise(F.regexp_extract(F.col("emp_length"), r"(\d+)", 1))
).withColumn(
    "emp_length",
    F.when(F.col("emp_length") == "", None)
    .otherwise(F.col("emp_length"))
    .cast("int")
)

display(selected_df.select("emp_length").distinct())

In [0]:
filtered_emp_length = selected_df.filter((col("emp_length").isNull()))
display(filtered_emp_length.count())

Changing all the null values to 0.

In [0]:
selected_df = selected_df.withColumn(
    "emp_length",
    F.when(F.col("emp_length").isNull(), 0).otherwise(F.col("emp_length"))
)

In [0]:
distinct_loan_status = selected_df.select("loan_status").distinct()
display(distinct_loan_status)

In [0]:
print("Oct-2015: ")
loan_status_filter = selected_df.filter((col("loan_status").contains("Oct-2015")))
display(loan_status_filter.select("loan_status").count())

print("\nDoes not meet the credit policy. Status:Charged Off: ")
loan_status_filter = selected_df.filter((col("loan_status").contains("Does not meet the credit policy. Status:Charged Off")))
display(loan_status_filter.select("loan_status").count())

We need to keep this corupt data into our mind because it might affect the overall result in the dashboard.

In [0]:
df_oct = selected_df.filter((col("loan_status").contains("Oct-2015")))
display(df_oct)

In [0]:
df_oct = selected_df.filter((col("loan_status").isNull()))
display(df_oct.count())

Dropped all the columns because all the data in rows were null and as well for all other columns as well.

In [0]:
selected_df = selected_df.filter(col("loan_status").isNotNull())

In [0]:
# Register the DataFrame as a temp view so we can query it with SQL
selected_df.createOrReplaceTempView("loans")

In [0]:
%sql
SELECT * FROM loans;

## Problem 1 — Top 5 States by Total Loan Amount

Find the **top 5 states** (`addr_state`) by **total loan amount**. For each state, show:
* `addr_state`
* total loan amount (alias `total_loan_amnt`)
* average interest rate (alias `avg_int_rate`, rounded to 2 decimals)
* number of loans (alias `num_loans`)

Order the results by `total_loan_amnt` descending.

In [0]:
%sql
SELECT 
    addr_state, 
    SUM(loan_amnt) AS total_loan_amnt,
    AVG(CAST(int_rate AS DECIMAL(10, 2))) AS avg_int_rate,
    COUNT(*) AS num_loans
FROM loans
GROUP BY addr_state
ORDER BY total_loan_amnt DESC;

## Problem 2 — Average Loan by Grade for High-Income Borrowers

For borrowers with `annual_inc > 100000`, find the **average loan amount** and **average interest rate** (rounded to 2 decimals) for each `grade`. Show only grades where the **number of loans is at least 50**. Order by `grade` ascending.

In [0]:
%sql

SELECT
    grade,
    ROUND(AVG(loan_amnt), 2) AS avg_loan_amnt,
    ROUND(AVG(int_rate), 2) AS avg_int_rate,
    COUNT(*) AS num_loans
FROM loans
WHERE annual_inc > 100000
GROUP BY grade
HAVING COUNT(*) >= 50
ORDER BY grade;

## Problem 3 — Top 3 Largest Loans per Grade

Using a **window function**, find the **top 3 largest loans** (by `loan_amnt`) for each `grade`. Show:
* `grade`
* `loan_amnt`
* `addr_state`
* `annual_inc`
* a ranking column (alias `loan_rank`)

Order the final results by `grade` ascending, then `loan_rank` ascending.

In [0]:
%sql

WITH ranked_loans AS (
    SELECT
        grade,
        loan_amnt,
        addr_state,
        annual_inc,
        ROW_NUMBER() OVER (PARTITION BY grade ORDER BY loan_amnt DESC) AS loan_rank
    FROM loans
)
SELECT *
FROM ranked_loans
WHERE loan_rank <= 3
ORDER BY grade, loan_rank;

## Problem 4 — Borrowers Above Their State's Average DTI

Find borrowers whose `dti` is **higher than the average `dti` of their state** (`addr_state`). Show:
* `addr_state`
* `loan_amnt`
* `grade`
* `dti`
* the state's average dti (alias `state_avg_dti`, rounded to 2 decimals)

Limit the output to 20 rows ordered by `dti` descending.

In [0]:
%sql
WITH state_avg AS (
    SELECT
        addr_state,
        loan_amnt,
        grade,
        dti,
        ROUND(AVG(TRY_CAST(dti AS DOUBLE)) OVER (PARTITION BY addr_state), 2) AS state_avg_dti
    FROM loans
    WHERE TRY_CAST(dti AS DOUBLE) < 999  -- 999 is a missing-value placeholder
)
SELECT *
FROM state_avg
WHERE TRY_CAST(dti AS DOUBLE) > state_avg_dti
ORDER BY TRY_CAST(dti AS DOUBLE) DESC
LIMIT 20;

In [0]:
display(selected_df)

In [0]:
print([repr(c) for c in selected_df.columns])

In [0]:
selected_df = selected_df.toDF(*[c.strip() for c in selected_df.columns])

In [0]:
df_cleaned = selected_df.withColumn(
    "dti",
    F.coalesce(F.col("dti").cast("double"), F.lit(0.0))
)

display(df_cleaned)

## Problem 5 — FICO Risk Buckets

Categorize borrowers into risk buckets using `fico_range_low`:
* **Low Risk**: FICO >= 700
* **Medium Risk**: FICO 630–699
* **High Risk**: FICO < 630

For each risk bucket, show:
* the risk label (alias `risk_bucket`)
* number of loans (alias `num_loans`)
* total loan amount (alias `total_loan_amnt`)
* average interest rate (alias `avg_int_rate`, rounded to 2 decimals)

Use a `CASE` expression and order by `avg_int_rate` descending.

In [0]:
%sql
SELECT
    CASE
        WHEN TRY_CAST(fico_range_low AS DOUBLE) >= 700 THEN 'Low Risk'
        WHEN TRY_CAST(fico_range_low AS DOUBLE) BETWEEN 630 AND 699 THEN 'Medium Risk'
        WHEN TRY_CAST(fico_range_low AS DOUBLE) < 630 THEN 'High Risk'
    END AS risk_bucket,
    COUNT(*) AS num_loans,
    SUM(loan_amnt) AS total_loan_amnt,
    ROUND(AVG(TRY_CAST(int_rate AS DOUBLE)), 2) AS avg_int_rate
FROM loans
WHERE fico_range_low IS NOT NULL
GROUP BY risk_bucket
ORDER BY avg_int_rate DESC;

In [0]:
from pyspark.sql import functions as F

df_narrow = df_cleaned.select("id", "loan_amnt").filter(F.col("loan_amnt") > 6000)
df_narrow.explain("extended")